# Triposplat Combined Notebook
This notebook integrates the Triposplat pipeline with an interactive UI for image upload, parameter control, and mesh download. All source files are imported from the repository.

In [ ]:
import sys, io, os, base64, tempfile
from pathlib import Path
import torch
from ipywidgets import FileUpload, IntSlider, FloatSlider, Dropdown, Button, HBox, VBox, Output
from IPython.display import display, HTML
from PIL import Image

# Ensure repository root is on sys.path
repo_path = Path.cwd()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Import project modules
import model
import triposplat
import run_gradio as rg
import inline_upload as iu

# Device selector (CPU/GPU)
device_dropdown = Dropdown(options=['cpu', 'cuda'], value='cuda' if torch.cuda.is_available() else 'cpu', description='Device')
DEVICE = torch.device(device_dropdown.value)
def on_device_change(change):
    global DEVICE
    DEVICE = torch.device(change['new'])
device_dropdown.observe(on_device_change, names='value')


In [ ]:
# UI Widgets
upload = FileUpload(accept='image/*', multiple=False)
steps_slider = IntSlider(value=20, min=1, max=100, description='Steps')
guidance_slider = FloatSlider(value=7.5, min=1.0, max=20.0, step=0.1, description='Guidance')
gauss_slider = IntSlider(value=256, min=64, max=1024, step=64, description='Gaussians')
run_button = Button(description='Generate')
download_button = Button(description='Download Mesh')
output = Output()

last_mesh = None  # placeholder for generated mesh object

def on_run_clicked(b):
    with output:
        output.clear_output()
        if not upload.value:
            print('Please upload an image.')
            return
        img_bytes = list(upload.value.values())[0]['content']
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        try:
            mesh = rg.generate(img,
                              steps=steps_slider.value,
                              guidance_scale=guidance_slider.value,
                              num_gaussians=gauss_slider.value,
                              device=DEVICE)
        except Exception as e:
            print('Generation error:', e)
            return
        print('Generation completed.')
        global last_mesh
        last_mesh = mesh
        if hasattr(mesh, 'preview_html'):
            display(HTML(mesh.preview_html()))
        else:
            print('Mesh generated. Use the download button to save it.')
run_button.on_click(on_run_clicked)

def on_download_clicked(b):
    if last_mesh is None:
        print('No mesh generated yet.')
        return
    with tempfile.NamedTemporaryFile(delete=False, suffix='.ply') as tmp:
        if hasattr(last_mesh, 'save'):
            last_mesh.save(tmp.name)
        else:
            tmp.write(last_mesh)
        tmp_path = tmp.name
    with open(tmp_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    html = f'<a download="mesh.ply" href="data:application/octet-stream;base64,{b64}">Download Mesh</a>'
    display(HTML(html))
download_button.on_click(on_download_clicked)

ui = VBox([device_dropdown, upload, steps_slider, guidance_slider, gauss_slider, HBox([run_button, download_button]), output])
display(ui)
